# Algorithmic Trading Strategy Optimization Using Genetic Algorithms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

data = pd.read_csv('trading_data.csv')
data['Date'] = pd.to_datetime(data['Date'])

data['SMA_20'] = data['Close'].rolling(window=20).mean()
data['SMA_50'] = data['Close'].rolling(window=50).mean()

delta    = data['Close'].diff()
gain     = delta.clip(lower=0)
loss     = -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0, 1e-9)
data['RSI'] = 100 - (100 / (1 + rs))
data['RSI'] = data['RSI'].replace([np.inf, -np.inf], np.nan)
flat_mask = (avg_gain == 0) & (avg_loss == 0)
data.loc[flat_mask, 'RSI'] = 50.0

ema_12 = data['Close'].ewm(span=12, adjust=False).mean()
ema_26 = data['Close'].ewm(span=26, adjust=False).mean()
data['MACD']        = ema_12 - ema_26
data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()

data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

prices      = data['Close'].values
sma_20      = data['SMA_20'].values
sma_50      = data['SMA_50'].values
rsi         = data['RSI'].values
macd        = data['MACD'].values
macd_signal = data['MACD_Signal'].values

print(f'Dataset loaded : {len(data)} usable rows')
print(f'Date range     : {data["Date"].iloc[0].date()} \u2192 {data["Date"].iloc[-1].date()}')
print(f'RSI range      : {rsi.min():.1f} \u2013 {rsi.max():.1f}')
print(f'MACD range     : {macd.min():.3f} \u2013 {macd.max():.3f}')
print()
data[['Close', 'SMA_20', 'SMA_50', 'RSI', 'MACD', 'MACD_Signal']].head()


## Individual Representation, Strategy Simulation & Fitness

In [ ]:
GENE_BOUNDS = {
    'rsi_buy'        : (20.0, 45.0),
    'rsi_sell'       : (55.0, 80.0),
    'macd_weight'    : (0.1,  2.0),
    'sma_weight'     : (0.1,  2.0),
    'buy_threshold'  : (0.5,  5.0),
    'sell_threshold' : (0.5,  5.0),
}


def random_individual():
    return {gene: np.random.uniform(lo, hi)
            for gene, (lo, hi) in GENE_BOUNDS.items()}


def simulate_strategy(individual, prices, sma_20, sma_50, rsi, macd, macd_signal,
                      return_positions=False):
    rsi_buy   = individual['rsi_buy']
    rsi_sell  = individual['rsi_sell']
    macd_w    = individual['macd_weight']
    sma_w     = individual['sma_weight']
    buy_thr   = individual['buy_threshold']
    sell_thr  = individual['sell_threshold']

    N         = len(prices)
    log_ret   = np.diff(np.log(prices))
    positions = np.zeros(N - 1)
    position  = 0

    for i in range(N - 1):
        sma_score  =  sma_w  if sma_20[i] > sma_50[i]     else -sma_w
        macd_score =  macd_w if macd[i]   > macd_signal[i] else -macd_w
        rsi_buy_s  =  1.0    if rsi[i]    < rsi_buy        else 0.0
        rsi_sell_s =  1.0    if rsi[i]    > rsi_sell       else 0.0

        buy_score  =  sma_score + macd_score + rsi_buy_s
        sell_score = -sma_score - macd_score + rsi_sell_s

        if   buy_score  >= buy_thr  and position <= 0:
            position = 1
        elif sell_score >= sell_thr and position >= 0:
            position = -1

        positions[i] = position

    strat_returns = positions[:-1] * log_ret[1:]

    if return_positions:
        return strat_returns, positions
    return strat_returns


def fitness(individual, prices, sma_20, sma_50, rsi, macd, macd_signal):
    strat_returns, positions = simulate_strategy(
        individual, prices, sma_20, sma_50, rsi, macd, macd_signal,
        return_positions=True)

    if np.count_nonzero(positions[:-1]) < 5:
        return -999.0

    std = strat_returns.std(ddof=1)
    if std < 1e-8:
        return -999.0
    sharpe = (strat_returns.mean() / std) * np.sqrt(252)
    return float(sharpe)


sample = random_individual()
print('Sample individual genes:')
for k, v in sample.items():
    lo, hi = GENE_BOUNDS[k]
    print(f'  {k:<20} = {v:.4f}   (bounds: [{lo}, {hi}])')
print(f'\nSample fitness (annualised Sharpe) : {fitness(sample, prices, sma_20, sma_50, rsi, macd, macd_signal):.6f}')


## Genetic Algorithm Operators

In [ ]:
def initialize_population(pop_size):
    return [random_individual() for _ in range(pop_size)]


def evaluate_population(population, prices, sma_20, sma_50, rsi, macd, macd_signal):
    return np.array([
        fitness(ind, prices, sma_20, sma_50, rsi, macd, macd_signal)
        for ind in population
    ])


def tournament_selection(population, fitness_scores, num_parents, k=3):
    parents  = []
    pop_size = len(population)
    k = min(k, pop_size)
    for _ in range(num_parents):
        contenders = np.random.choice(pop_size, size=k, replace=False)
        winner_idx = contenders[np.argmax(fitness_scores[contenders])]
        parents.append(population[winner_idx])
    return parents


def uniform_crossover(parent1, parent2):
    child = {}
    for gene in GENE_BOUNDS:
        child[gene] = parent1[gene] if np.random.rand() < 0.5 else parent2[gene]
    return child


def gaussian_mutation(individual, mutation_rate=0.2, mutation_strength=0.15):
    mutant = individual.copy()
    for gene, (lo, hi) in GENE_BOUNDS.items():
        if np.random.rand() < mutation_rate:
            noise = np.random.normal(0, mutation_strength * (hi - lo))
            mutant[gene] = float(np.clip(mutant[gene] + noise, lo, hi))
    return mutant


def elitism(population, fitness_scores, n_elite):
    elite_indices = np.argsort(fitness_scores)[-n_elite:]
    return [population[i] for i in elite_indices]


print('GA operators defined: initialize_population, evaluate_population,')
print('tournament_selection, uniform_crossover, gaussian_mutation, elitism')


## Main Genetic Algorithm

In [ ]:
def genetic_algorithm(
    prices, sma_20, sma_50, rsi, macd, macd_signal,
    num_generations   = 50,
    pop_size          = 100,
    num_parents       = 30,
    n_elite           = 5,
    mutation_rate     = 0.2,
    mutation_strength = 0.15,
    tournament_k      = 3,
    verbose           = True,
):
    population           = initialize_population(pop_size)
    history              = {'best': [], 'mean': [], 'worst': []}
    best_ever_fitness    = -np.inf
    best_ever_individual = None

    if verbose:
        print(f'{"Gen":>6}  {"Best":>10}  {"Mean":>10}  {"Worst (valid)":>14}')
        print('-' * 49)

    for gen in range(num_generations):
        fitness_scores = evaluate_population(
            population, prices, sma_20, sma_50, rsi, macd, macd_signal)

        valid_mask   = fitness_scores > -999
        valid_scores = fitness_scores[valid_mask]

        gen_best  = fitness_scores.max()
        gen_mean  = valid_scores.mean() if valid_scores.size else -999.0
        gen_worst = valid_scores.min()  if valid_scores.size else -999.0

        history['best'].append(gen_best)
        history['mean'].append(gen_mean)
        history['worst'].append(gen_worst)

        if gen_best > best_ever_fitness:
            best_ever_fitness    = gen_best
            best_ever_individual = population[int(np.argmax(fitness_scores))].copy()

        if verbose:
            print(f'{gen+1:>6}  {gen_best:>+10.5f}  {gen_mean:>+10.5f}  {gen_worst:>+14.5f}')

        elite_pool = elitism(population, fitness_scores, n_elite)
        parents    = tournament_selection(
            population, fitness_scores, num_parents, k=tournament_k)

        n_offspring = pop_size - n_elite
        n_parents   = len(parents)
        offspring   = []
        for _ in range(n_offspring):
            if n_parents >= 2:
                idx1, idx2 = np.random.choice(n_parents, size=2, replace=False)
            else:
                idx1, idx2 = 0, 0
            child = uniform_crossover(parents[idx1], parents[idx2])
            child = gaussian_mutation(child, mutation_rate, mutation_strength)
            offspring.append(child)

        population = elite_pool + offspring

    final_scores   = evaluate_population(
        population, prices, sma_20, sma_50, rsi, macd, macd_signal)
    final_best_idx = int(np.argmax(final_scores))
    if final_scores[final_best_idx] > best_ever_fitness:
        best_ever_fitness    = final_scores[final_best_idx]
        best_ever_individual = population[final_best_idx].copy()

    if best_ever_fitness <= -999.0:
        raise ValueError(
            'GA failed: no valid strategy found across all generations. '
            'Check dataset length (>60 rows recommended), positive Close prices, '
            'and GENE_BOUNDS.'
        )

    return best_ever_individual, history


print('Starting Genetic Algorithm...\n')

best_individual, history = genetic_algorithm(
    prices, sma_20, sma_50, rsi, macd, macd_signal,
    num_generations   = 50,
    pop_size          = 100,
    num_parents       = 30,
    n_elite           = 5,
    mutation_rate     = 0.2,
    mutation_strength = 0.15,
    tournament_k      = 3,
    verbose           = True,
)

print('\n' + '=' * 45)
print('BEST INDIVIDUAL FOUND')
print('=' * 45)
best_fitness_val = fitness(best_individual, prices, sma_20, sma_50, rsi, macd, macd_signal)
print(f'Fitness (Annualised Sharpe) : {best_fitness_val:.6f}\n')
print('Genes:')
for gene, val in best_individual.items():
    lo, hi = GENE_BOUNDS[gene]
    print(f'  {gene:<20} = {val:.4f}   (bounds: [{lo}, {hi}])')


## Backtest the Best Strategy

In [ ]:
def backtest(individual, prices, sma_20, sma_50, rsi, macd, macd_signal, dates):
    strat_returns, positions = simulate_strategy(
        individual, prices, sma_20, sma_50, rsi, macd, macd_signal,
        return_positions=True)

    log_ret          = np.diff(np.log(prices))
    strat_ret_padded = np.concatenate([[0.0], strat_returns])
    strat_equity     = np.concatenate([[1.0], np.cumprod(np.exp(strat_ret_padded))])
    bnh_equity       = np.concatenate([[1.0], np.cumprod(np.exp(log_ret))])

    active_pos = positions[:-1]
    prev_pos   = np.concatenate([[0], active_pos[:-1]])
    n_trades   = int(((prev_pos == 0) & (active_pos != 0)).sum())

    df = pd.DataFrame({
        'Date'            : dates,
        'Close'           : prices,
        'Strategy_Equity' : strat_equity,
        'BuyHold_Equity'  : bnh_equity,
        'Strategy_Returns': np.concatenate([strat_ret_padded, [0.0]]),
    })
    return df, n_trades


bt_df, n_trades = backtest(
    best_individual, prices, sma_20, sma_50, rsi, macd, macd_signal,
    data['Date'].values
)

total_return_strat = bt_df['Strategy_Equity'].iloc[-1] - 1
total_return_bnh   = bt_df['BuyHold_Equity'].iloc[-1]  - 1

print('BACKTEST RESULTS')
print(f'  Strategy total return : {total_return_strat:+.2%}')
print(f'  Buy-and-hold return   : {total_return_bnh:+.2%}')
print(f'  Number of trades      : {n_trades}')
print(f'  Equity curve start    : {bt_df["Strategy_Equity"].iloc[0]:.4f}  (should be 1.0000)')


## Visualisations

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 14))

ax = axes[0]
ax.plot(history['best'],  label='Best',             linewidth=2,   color='steelblue')
ax.plot(history['mean'],  label='Mean',             linewidth=1.5, color='darkorange', linestyle='--')
ax.plot(history['worst'], label='Worst (valid only)', linewidth=1, color='tomato',    linestyle=':')
ax.set_title('GA Fitness Evolution (Annualised Sharpe Ratio) per Generation', fontsize=13)
ax.set_xlabel('Generation')
ax.set_ylabel('Annualised Sharpe Ratio')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(bt_df['Date'], bt_df['Strategy_Equity'], label='Optimised Strategy',
        linewidth=2, color='steelblue')
ax.plot(bt_df['Date'], bt_df['BuyHold_Equity'],  label='Buy & Hold',
        linewidth=1.5, color='gray', linestyle='--')
ax.set_title('Equity Curve: Optimised Strategy vs Buy-and-Hold', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value (normalised to 1.0)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(data['Date'], data['Close'],  label='Close',  linewidth=1,   color='black')
ax.plot(data['Date'], data['SMA_20'], label='SMA 20', linewidth=1.5, color='steelblue')
ax.plot(data['Date'], data['SMA_50'], label='SMA 50', linewidth=1.5, color='darkorange')
ax.set_title('Close Price with SMA Overlays', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ga_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to ga_results.png')
